Oskar Sagstuen Nysæther, oskarsn, oskarsn@stud.ntnu.no
Eirin Sofie Aspehjell Rishaug, esrishau, esrishau@stud.ntnu.no

# Innledning


## Problemstilling

*Hva er problemstillingen?*


## Valg av metoder
Multippel lineær regresjon og Random Forest.
*Gi en begrunnelse for hvilke metoder vi har valgt.*


# Data


## Beskrivelse av datasettet

*Beskriv datasettet. Hva er variablene?*


In [ ]:
import pandas as pd

df = pd.read_csv("Pakistan Largest Ecommerce Dataset.csv", low_memory=False)

# The file contains empty trailing columns and rows
df = df.dropna(axis=1, how="all").dropna(how="all")
print(df.shape)
df.head()


## Preprosessering

*Er det noen variabler som trenger et nytt navn, eller å dummy-kodes e.l., eller som bør utelates?*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
)

RANDOM_STATE = 42
N_BOOTSTRAP = 30


In [ ]:
df = df.rename(columns=lambda name: name.strip())
df["created_at"] = pd.to_datetime(df["created_at"], format="%m/%d/%Y")
df["category_name_1"] = df["category_name_1"].replace("\\N", np.nan).fillna("Unknown")


In [ ]:
# Refunds happen after delivery and are a different outcome than a cancellation
OUTCOMES = {"complete": 0, "canceled": 1}

# From May 2018 "complete" is replaced by "received", so later orders are not comparable
LAST_ORDER_DATE = pd.Timestamp("2018-04-30")

lines = df[df["status"].isin(OUTCOMES.keys()) & (df["created_at"] <= LAST_ORDER_DATE)].copy()
lines["Cancelled"] = lines["status"].map(OUTCOMES)
print("Order lines:", len(lines))


In [ ]:
# Status is set per order (all lines of an order share it), so each order is one observation
lines["line_value"] = lines["price"] * lines["qty_ordered"]
main_category = lines.sort_values("line_value").groupby("increment_id")["category_name_1"].last()

orders = lines.groupby("increment_id").agg(
    created_at=("created_at", "first"),
    payment_method=("payment_method", "first"),
    grand_total=("grand_total", "first"),
    gross_value=("line_value", "sum"),
    discount=("discount_amount", "sum"),
    items=("item_id", "size"),
    quantity=("qty_ordered", "sum"),
    categories=("category_name_1", "nunique"),
    max_price=("price", "max"),
    Cancelled=("Cancelled", "first"),
)
orders["main_category"] = main_category

print("Orders:", len(orders), "Cancelled share:", round(orders["Cancelled"].mean(), 3))


In [ ]:
orders["log_grand_total"] = np.log1p(orders["grand_total"].clip(lower=0))
orders["log_gross_value"] = np.log1p(orders["gross_value"].clip(lower=0))
orders["log_max_price"] = np.log1p(orders["max_price"].clip(lower=0))
orders["log_quantity"] = np.log1p(orders["quantity"])
orders["discount_share"] = (orders["discount"] / orders["gross_value"].replace(0, np.nan)).fillna(0).clip(0, 1)
orders["weekday"] = orders["created_at"].dt.day_name()
orders["month"] = orders["created_at"].dt.month_name()

TOP_PAYMENT_METHODS = orders["payment_method"].value_counts().index[:8]
orders["payment_group"] = orders["payment_method"].where(orders["payment_method"].isin(TOP_PAYMENT_METHODS), "Other")

dummies = pd.get_dummies(
    orders[["payment_group", "main_category", "weekday", "month"]], drop_first=True, dtype=int
)
orders = pd.concat([orders, dummies], axis=1)

FEATURES = [
    "log_grand_total", "log_gross_value", "log_max_price", "discount_share",
    "items", "log_quantity", "categories",
] + list(dummies.columns)
TARGET = "Cancelled"

orders[FEATURES].describe().T


In [ ]:
# The last period is set aside before any training and is only used for the final evaluation
VALIDATION_START = pd.Timestamp("2018-01-01")

development = orders[orders["created_at"] < VALIDATION_START]
validation = orders[orders["created_at"] >= VALIDATION_START]

for name, part in [("Development", development), ("Validation", validation)]:
    print(
        f"{name:<12} {part['created_at'].min().date()} - {part['created_at'].max().date()}"
        f"  orders: {len(part):>7}  cancelled: {part[TARGET].mean():.1%}"
    )


In [ ]:
def evaluate(actual, probability, threshold=0.5):
    predicted = (probability >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(actual, probability),
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted),
        "Recall": recall_score(actual, predicted),
    }


In [ ]:
def bootstrap_out_of_bag(make_model, data, describe, n_bootstrap=N_BOOTSTRAP):
    """Fit the model on bootstrap samples of the data and evaluate each fit on the orders
    left out of that sample (out-of-bag). `describe` extracts e.g. coefficients from each fit."""
    rng = np.random.default_rng(RANDOM_STATE)
    scores, descriptions = [], []
    for _ in range(n_bootstrap):
        in_bag = rng.integers(0, len(data), len(data))
        out_of_bag = np.setdiff1d(np.arange(len(data)), in_bag)
        model = make_model().fit(data[FEATURES].iloc[in_bag], data[TARGET].iloc[in_bag])
        probability = model.predict_proba(data[FEATURES].iloc[out_of_bag])[:, 1]
        scores.append(evaluate(data[TARGET].iloc[out_of_bag], probability))
        descriptions.append(describe(model))
    return pd.DataFrame(scores), pd.DataFrame(descriptions)


def interval(samples):
    """Median and 95% bootstrap interval for each column."""
    return samples.quantile([0.025, 0.5, 0.975]).T.set_axis(["2.5%", "Median", "97.5%"], axis=1)


## Eksplorativ dataanalyse

*Gi leseren et inntrykk av datasettet ved hjelp av relevante grafer.*


# Læringsmetode 1: Logistisk regresjon


## Grunnprinsipper

*Beskriv grunnprinsippene for metoden, evt. med noen matematiske formler.*


## Anvendelse og resultater

*Anvend metoden på datasettet og vis resultatene. Kommenter koden tydelig, og kommenter og tolk resultatene fortløpende.*


In [ ]:
def make_logistic_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


logistic_scores, logistic_coefficients = bootstrap_out_of_bag(
    make_logistic_model, development, describe=lambda model: pd.Series(model[-1].coef_[0], index=FEATURES)
)
print("Out-of-bag performance on the development period:")
interval(logistic_scores)


In [ ]:
coefficient_intervals = interval(logistic_coefficients)
coefficient_intervals["Excludes zero"] = (coefficient_intervals["2.5%"] > 0) | (coefficient_intervals["97.5%"] < 0)
coefficient_intervals.sort_values("Median", key=abs, ascending=False).head(15)


In [ ]:
logistic_model = make_logistic_model().fit(development[FEATURES], development[TARGET])
logistic_probability = logistic_model.predict_proba(validation[FEATURES])[:, 1]
print("Validation:", evaluate(validation[TARGET], logistic_probability))


# Læringsmetode 2: Random forest


## Grunnprinsipper

*Beskriv grunnprinsippene for metoden, evt. med noen matematiske formler.*


## Anvendelse og resultater

*Anvend metoden på datasettet og vis resultatene. Kommenter koden tydelig, og kommenter og tolk resultatene fortløpende.*


In [ ]:
# Out-of-bag orders come from the same period as the training data, so they reward patterns that do not
# carry over in time. The parameters are therefore chosen by training on the first year of the development
# period and scoring on the last six months.
TUNING_SPLIT = pd.Timestamp("2017-07-01")
tuning_train = development[development["created_at"] < TUNING_SPLIT]
tuning_test = development[development["created_at"] >= TUNING_SPLIT]

PARAMETER_GRID = [
    {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}
    for max_depth in [6, 10, None]
    for min_samples_leaf in [20, 100, 500]
]

tuning_results = []
for parameters in PARAMETER_GRID:
    forest = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE, **parameters)
    forest.fit(tuning_train[FEATURES], tuning_train[TARGET])
    later_auc = roc_auc_score(tuning_test[TARGET], forest.predict_proba(tuning_test[FEATURES])[:, 1])
    tuning_results.append({**parameters, "later_period_AUC": later_auc})

tuning_results = pd.DataFrame(tuning_results).sort_values("later_period_AUC", ascending=False)
best = tuning_results.iloc[0]
best_parameters = {
    "max_depth": None if pd.isna(best["max_depth"]) else int(best["max_depth"]),
    "min_samples_leaf": int(best["min_samples_leaf"]),
}
print("Best parameters:", best_parameters)
tuning_results


In [ ]:
# Every tree is trained on its own bootstrap sample; the orders a tree never saw give the out-of-bag score
forest = RandomForestClassifier(
    n_estimators=300, oob_score=True, n_jobs=-1, random_state=RANDOM_STATE, **best_parameters
)
forest.fit(development[FEATURES], development[TARGET])
forest_out_of_bag_auc = roc_auc_score(development[TARGET], forest.oob_decision_function_[:, 1])
forest_probability = forest.predict_proba(validation[FEATURES])[:, 1]

print("Out-of-bag AUC (development):", round(forest_out_of_bag_auc, 3))
print("Validation:", evaluate(validation[TARGET], forest_probability))


In [ ]:
importances = pd.Series(forest.feature_importances_, index=FEATURES).sort_values().tail(15)

fig, ax = plt.subplots(figsize=(8, 7))
importances.plot.barh(ax=ax)
ax.set(title="Random forest feature importance", xlabel="Mean decrease in impurity")
plt.show()


# Sammenligning og drøfting


## Tolkning av resultatene

*Tolk resultatene til de to metodene i lys av hverandre. Er den ene tydelig bedre enn den andre? Har de forskjellige styrker og svakheter?*


In [ ]:
# Simple rule the models must beat: the cancelled share of each payment method in the development data
payment_cancellation_rate = development.groupby("payment_group")[TARGET].mean()
baseline_probability = (
    validation["payment_group"].map(payment_cancellation_rate).fillna(development[TARGET].mean()).to_numpy()
)

probabilities = {
    "Baseline: payment method": baseline_probability,
    "Logistic regression": logistic_probability,
    "Random forest": forest_probability,
}
pd.DataFrame({name: evaluate(validation[TARGET], p) for name, p in probabilities.items()}).T


In [ ]:
N_VALIDATION_BOOTSTRAP = 500

rng = np.random.default_rng(RANDOM_STATE)
actual = validation[TARGET].to_numpy()
validation_samples = []
for _ in range(N_VALIDATION_BOOTSTRAP):
    sample = rng.integers(0, len(actual), len(actual))
    logistic_auc = roc_auc_score(actual[sample], logistic_probability[sample])
    forest_auc = roc_auc_score(actual[sample], forest_probability[sample])
    validation_samples.append({
        "Logistic regression AUC": logistic_auc,
        "Random forest AUC": forest_auc,
        "Difference (forest - logistic)": forest_auc - logistic_auc,
    })

interval(pd.DataFrame(validation_samples))


In [ ]:
# Random out-of-bag orders share the training period; the later development months and the validation period lie in the future
logistic_later_model = make_logistic_model().fit(tuning_train[FEATURES], tuning_train[TARGET])
forest_later_model = RandomForestClassifier(
    n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE, **best_parameters
).fit(tuning_train[FEATURES], tuning_train[TARGET])

pd.DataFrame({
    "Out-of-bag AUC (development)": [logistic_scores["ROC-AUC"].median(), forest_out_of_bag_auc],
    "Later development months AUC": [
        roc_auc_score(tuning_test[TARGET], logistic_later_model.predict_proba(tuning_test[FEATURES])[:, 1]),
        roc_auc_score(tuning_test[TARGET], forest_later_model.predict_proba(tuning_test[FEATURES])[:, 1]),
    ],
    "Validation AUC": [roc_auc_score(actual, logistic_probability), roc_auc_score(actual, forest_probability)],
}, index=["Logistic regression", "Random forest"])


In [ ]:
fig, (roc_ax, gains_ax) = plt.subplots(1, 2, figsize=(13, 5))

for name, probability in probabilities.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(validation[TARGET], probability)
    roc_ax.plot(false_positive_rate, true_positive_rate, label=name)

    ranked_actual = validation[TARGET].to_numpy()[np.argsort(-probability, kind="stable")]
    share_checked = np.arange(1, len(ranked_actual) + 1) / len(ranked_actual)
    gains_ax.plot(share_checked, np.cumsum(ranked_actual) / ranked_actual.sum(), label=name)

roc_ax.plot([0, 1], [0, 1], "k--", label="Random")
roc_ax.set(title="ROC curve (validation)", xlabel="False positive rate", ylabel="True positive rate")
gains_ax.plot([0, 1], [0, 1], "k--", label="Random")
gains_ax.set(title="Cumulative gains (validation)", xlabel="Share of orders checked", ylabel="Share of cancellations found")
roc_ax.legend()
gains_ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, probability) in zip(axes, [("Logistic regression", logistic_probability), ("Random forest", forest_probability)]):
    ConfusionMatrixDisplay(
        confusion_matrix(validation[TARGET], (probability >= 0.5).astype(int)),
        display_labels=["Completed", "Cancelled"],
    ).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(name)
plt.show()


## Feilanalyse

*Der resultatene ikke ble så gode som vi håpet, hva kan det skyldes? Går det an å se på resultatene i detalj for å forstå mer av hva som evt. gikk galt?*


In [ ]:
validation_errors = validation.assign(logistic=logistic_probability, forest=forest_probability)

monthly = validation_errors.groupby(validation_errors["created_at"].dt.to_period("M")).agg(
    orders=(TARGET, "size"),
    actual_cancelled=(TARGET, "mean"),
    logistic_predicted=("logistic", "mean"),
    forest_predicted=("forest", "mean"),
)
monthly["logistic_AUC"] = validation_errors.groupby(validation_errors["created_at"].dt.to_period("M")).apply(
    lambda month: roc_auc_score(month[TARGET], month["logistic"])
)
monthly["forest_AUC"] = validation_errors.groupby(validation_errors["created_at"].dt.to_period("M")).apply(
    lambda month: roc_auc_score(month[TARGET], month["forest"])
)
monthly


# Konklusjon


## Oppsummering

*Hva har vi gjort? Hvor bra gikk det? Hva slags lærdommer kan vi ta med oss fra det vi har gjort?*


## Videre arbeid

*Hva slags forbedringer kan vi se for oss av det vi har gjort – hvis vi hadde hatt mye mer tid og ressurser enn vi har?*
